## 03 — Feature Engineering

1. Train/test split by season
2. Compute ratio features on train and test separately
3. Fill denominator-zero NaNs using train means only
4. Drop denominator columns
5. Apply 5-match rolling window with shift(1) per player to prevent data leakage
6. Drop first-appearance NaN rows from rolling shift
7. Map position_id to position groups via POSITION_MAP
8. Fit RobustScaler per position group on train only, apply to train and test
9. Clip GK aerial_win_rate and ground_duel_win_rate at ±5 std
10. Save scaled parquets and scalers to data/models/

In [117]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('data/processed')

outfield = pd.read_parquet(PROCESSED_DIR / 'outfield_clean.parquet')
gk       = pd.read_parquet(PROCESSED_DIR / 'gk_clean.parquet')

ID_COLS = [
    'match_id', 'round', 'match_date', 'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name', 'minutes_played',
    'shirt_number', 'position_id', 'is_goalkeeper', 'season',
    # match context features
    'home_away', 'opponent', 'team_goals_scored', 'opp_goals_scored', 'result'
]

print(f'Outfield : {outfield.shape[0]:,} rows x {outfield.shape[1]} cols')
print(f'GK       : {gk.shape[0]:,} rows x {gk.shape[1]} cols')
print(f'\nOutfield seasons : {sorted(outfield["season"].unique())}')
print(f'GK seasons       : {sorted(gk["season"].unique())}')
print(f'\nOutfield dtypes:\n{outfield.dtypes.value_counts()}')

Outfield : 30,329 rows x 55 cols
GK       : 3,042 rows x 45 cols

Outfield seasons : ['2021_2022', '2022_2023', '2023_2024', '2024_2025']
GK seasons       : ['2021_2022', '2022_2023', '2023_2024', '2024_2025']

Outfield dtypes:
float64                40
object                  8
int64                   5
datetime64[ns, UTC]     1
bool                    1
Name: count, dtype: int64


In [118]:
# ── Train/test split by season ─────────────────────────────────
TRAIN_SEASONS = ['2021_2022', '2022_2023', '2023_2024']
TEST_SEASON   = ['2024_2025']

outfield_train = outfield[outfield['season'].isin(TRAIN_SEASONS)].copy()
outfield_test  = outfield[outfield['season'].isin(TEST_SEASON)].copy()

gk_train = gk[gk['season'].isin(TRAIN_SEASONS)].copy()
gk_test  = gk[gk['season'].isin(TEST_SEASON)].copy()

print(f'Outfield train : {len(outfield_train):,} rows ({len(outfield_train)/len(outfield)*100:.1f}%)')
print(f'Outfield test  : {len(outfield_test):,} rows ({len(outfield_test)/len(outfield)*100:.1f}%)')
print(f'\nGK train : {len(gk_train):,} rows ({len(gk_train)/len(gk)*100:.1f}%)')
print(f'GK test  : {len(gk_test):,} rows ({len(gk_test)/len(gk)*100:.1f}%)')

Outfield train : 22,780 rows (75.1%)
Outfield test  : 7,549 rows (24.9%)

GK train : 2,282 rows (75.0%)
GK test  : 760 rows (25.0%)


In [119]:
def compute_ratios(df):
    """Compute ratio features for outfield players."""
    df = df.copy()
    df['pass_accuracy']        = df['accurate_passes']    / df['accurate_passes_total']
    df['long_ball_accuracy']   = df['long_balls_accurate'] / df['long_balls_accurate_total']
    df['cross_accuracy']       = df['accurate_crosses']    / df['accurate_crosses_total']
    df['dribble_success_rate'] = df['dribbles_succeeded']  / df['dribbles_succeeded_total']
    df['aerial_win_rate']      = df['aerials_won']         / df['aerials_won_total']
    df['ground_duel_win_rate'] = df['ground_duels_won']    / df['ground_duels_won_total']
    df['shot_accuracy']        = df['ShotsOnTarget']       / (df['ShotsOnTarget'] + df['ShotsOffTarget'])
    return df

def compute_ratios_gk(df):
    """Compute ratio features for goalkeepers."""
    df = df.copy()
    df['pass_accuracy']      = df['accurate_passes']    / df['accurate_passes_total']
    df['long_ball_accuracy'] = df['long_balls_accurate'] / df['long_balls_accurate_total']
    df['aerial_win_rate']      = df['aerials_won']         / df['aerials_won_total']
    df['ground_duel_win_rate'] = df['ground_duels_won']    / df['ground_duels_won_total']
    df['save_rate']          = df['saves']               / (df['saves'] + df['goals_conceded'])
    return df

In [120]:
outfield_train = compute_ratios(outfield_train)
outfield_test  = compute_ratios(outfield_test)
gk_train       = compute_ratios_gk(gk_train)
gk_test        = compute_ratios_gk(gk_test)

# Check NaNs produced
RATIO_COLS_OUTFIELD = [
    'pass_accuracy', 'long_ball_accuracy', 'cross_accuracy',
    'dribble_success_rate', 'aerial_win_rate', 'ground_duel_win_rate', 'shot_accuracy'
]
RATIO_COLS_GK = ['pass_accuracy', 'long_ball_accuracy','aerial_win_rate', 'ground_duel_win_rate', 'save_rate']

print('=== OUTFIELD TRAIN — NaNs from division ===')
for col in RATIO_COLS_OUTFIELD:
    n = outfield_train[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(outfield_train)*100:.1f}%)')

print('\n=== OUTFIELD TEST — NaNs from division ===')
for col in RATIO_COLS_OUTFIELD:
    n = outfield_test[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(outfield_test)*100:.1f}%)')

print('\n=== GK TRAIN — NaNs from division ===')
for col in RATIO_COLS_GK:
    n = gk_train[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(gk_train)*100:.1f}%)')

print('\n=== GK TEST — NaNs from division ===')
for col in RATIO_COLS_GK:
    n = gk_test[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(gk_test)*100:.1f}%)')

=== OUTFIELD TRAIN — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy         4151 NaNs (18.2%)
  cross_accuracy            10843 NaNs (47.6%)
  dribble_success_rate       9494 NaNs (41.7%)
  aerial_win_rate            4288 NaNs (18.8%)
  ground_duel_win_rate        546 NaNs (2.4%)
  shot_accuracy             11766 NaNs (51.7%)

=== OUTFIELD TEST — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy         1460 NaNs (19.3%)
  cross_accuracy             3519 NaNs (46.6%)
  dribble_success_rate       3159 NaNs (41.8%)
  aerial_win_rate            1446 NaNs (19.2%)
  ground_duel_win_rate        184 NaNs (2.4%)
  shot_accuracy              3937 NaNs (52.2%)

=== GK TRAIN — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy           10 NaNs (0.4%)
  aerial_win_rate            1723 NaNs (75.5%)
  ground_duel_win_rate       1841 NaNs (80.7%)
  save_rate                    61

In [121]:
# ── Compute means from TRAINING data only ─────────────────────
outfield_ratio_means = outfield_train[RATIO_COLS_OUTFIELD].mean()
gk_ratio_means       = gk_train[RATIO_COLS_GK].mean()

print('=== OUTFIELD — training means used for filling ===')
for col, val in outfield_ratio_means.items():
    print(f'  {col:<25} {val:.4f}')

print('\n=== GK — training means used for filling ===')
for col, val in gk_ratio_means.items():
    print(f'  {col:<25} {val:.4f}')

# ── Fill NaNs in BOTH train and test using train means only ───
outfield_train[RATIO_COLS_OUTFIELD] = outfield_train[RATIO_COLS_OUTFIELD].fillna(outfield_ratio_means)
outfield_test[RATIO_COLS_OUTFIELD]  = outfield_test[RATIO_COLS_OUTFIELD].fillna(outfield_ratio_means)
gk_train[RATIO_COLS_GK]             = gk_train[RATIO_COLS_GK].fillna(gk_ratio_means)
gk_test[RATIO_COLS_GK]              = gk_test[RATIO_COLS_GK].fillna(gk_ratio_means)


=== OUTFIELD — training means used for filling ===
  pass_accuracy             0.8114
  long_ball_accuracy        0.4769
  cross_accuracy            0.2286
  dribble_success_rate      0.4967
  aerial_win_rate           0.4828
  ground_duel_win_rate      0.5070
  shot_accuracy             0.4722

=== GK — training means used for filling ===
  pass_accuracy             0.6734
  long_ball_accuracy        0.3687
  aerial_win_rate           0.9798
  ground_duel_win_rate      0.7680
  save_rate                 0.6726


In [122]:
# ── Drop denominator columns — no longer needed ───────────────
DENOM_COLS_OUTFIELD = [
    'accurate_passes_total', 'long_balls_accurate_total', 'accurate_crosses_total',
    'dribbles_succeeded_total', 'aerials_won_total', 'ground_duels_won_total'
]
DENOM_COLS_GK = [
    'accurate_passes_total', 'long_balls_accurate_total',
    'aerials_won_total', 'ground_duels_won_total'
]

outfield_train = outfield_train.drop(columns=DENOM_COLS_OUTFIELD)
outfield_test  = outfield_test.drop(columns=DENOM_COLS_OUTFIELD)
gk_train       = gk_train.drop(columns=DENOM_COLS_GK)
gk_test        = gk_test.drop(columns=DENOM_COLS_GK)

# ── Confirm no NaNs remain in ratio cols ──────────────────────
print('\n=== NaN check after filling ===')
print(f'Outfield train ratio NaNs : {outfield_train[RATIO_COLS_OUTFIELD].isna().sum().sum()}')
print(f'Outfield test ratio NaNs  : {outfield_test[RATIO_COLS_OUTFIELD].isna().sum().sum()}')
print(f'GK train ratio NaNs       : {gk_train[RATIO_COLS_GK].isna().sum().sum()}')
print(f'GK test ratio NaNs        : {gk_test[RATIO_COLS_GK].isna().sum().sum()}')

print(f'\nOutfield train : {outfield_train.shape[0]:,} rows x {outfield_train.shape[1]} cols')
print(f'Outfield test  : {outfield_test.shape[0]:,} rows x {outfield_test.shape[1]} cols')
print(f'GK train       : {gk_train.shape[0]:,} rows x {gk_train.shape[1]} cols')
print(f'GK test        : {gk_test.shape[0]:,} rows x {gk_test.shape[1]} cols')


=== NaN check after filling ===
Outfield train ratio NaNs : 0
Outfield test ratio NaNs  : 0
GK train ratio NaNs       : 0
GK test ratio NaNs        : 0

Outfield train : 22,780 rows x 56 cols
Outfield test  : 7,549 rows x 56 cols
GK train       : 2,282 rows x 46 cols
GK test        : 760 rows x 46 cols


In [123]:
WINDOW = 5

# Columns to apply rolling window to
ROLLING_COLS_OUTFIELD = [c for c in outfield_train.columns
                         if c not in ID_COLS + ['rating_title', 'season', 'round', 'match_date']]

ROLLING_COLS_GK = [c for c in gk_train.columns
                   if c not in ID_COLS + ['rating_title', 'season', 'round', 'match_date']]

print(f'Rolling window cols — Outfield : {len(ROLLING_COLS_OUTFIELD)}')
print(f'Rolling window cols — GK       : {len(ROLLING_COLS_GK)}')


def apply_rolling(df, rolling_cols, window=5):
    """
    For each stat column:
      1. Sort by player and date so matches are in chronological order
      2. Shift by 1 so current match is never included in its own window
      3. Compute rolling mean over the last N matches
      4. Replace original column with the rolling version
    """
    df = df.copy()
    df = df.sort_values(['player_id', 'match_date']).reset_index(drop=True)

    for col in rolling_cols:
        df[col] = (
            df.groupby('player_id')[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
    return df

Rolling window cols — Outfield : 36
Rolling window cols — GK       : 26


In [124]:
print('Applying rolling windows to outfield train...')
outfield_train_rolled = apply_rolling(outfield_train, ROLLING_COLS_OUTFIELD, WINDOW)
print('Applying rolling windows to outfield test...')
outfield_test_rolled  = apply_rolling(outfield_test,  ROLLING_COLS_OUTFIELD, WINDOW)

print('Applying rolling windows to GK train...')
gk_train_rolled = apply_rolling(gk_train, ROLLING_COLS_GK, WINDOW)
print('Applying rolling windows to GK test...')
gk_test_rolled  = apply_rolling(gk_test,  ROLLING_COLS_GK, WINDOW)

# check one player's accurate_passes column before and after
sample_player = outfield_train['player_id'].value_counts().index[3]
print(f'\n=== Sample player {sample_player} — accurate_passes before vs after rolling ===')
before = outfield_train[outfield_train['player_id'] == sample_player][['match_date', 'round', 'accurate_passes']].head(8)
after  = outfield_train_rolled[outfield_train_rolled['player_id'] == sample_player][['match_date', 'round', 'accurate_passes']].head(8)

print('\nBefore:')
print(before.to_string(index=False))
print('\nAfter (5-match rolling avg):')
print(after.to_string(index=False))


Applying rolling windows to outfield train...
Applying rolling windows to outfield test...
Applying rolling windows to GK train...
Applying rolling windows to GK test...

=== Sample player 1120224 — accurate_passes before vs after rolling ===

Before:
               match_date  round  accurate_passes
2021-08-14 14:00:00+00:00      1              9.0
2021-08-21 14:00:00+00:00      2             35.0
2021-08-28 14:00:00+00:00      3             25.0
2021-09-11 11:30:00+00:00      4             14.0
2021-09-18 14:00:00+00:00      5             28.0
2021-09-27 19:00:00+00:00      6             14.0
2021-10-03 13:00:00+00:00      7             27.0
2021-10-18 19:00:00+00:00      8             30.0

After (5-match rolling avg):
               match_date  round  accurate_passes
2021-08-14 14:00:00+00:00      1              NaN
2021-08-21 14:00:00+00:00      2             9.00
2021-08-28 14:00:00+00:00      3            22.00
2021-09-11 11:30:00+00:00      4            23.00
2021-09-18 14:00:0

In [125]:
print(f'Outfield train NaNs : {outfield_train_rolled[ROLLING_COLS_OUTFIELD].isna().sum().sum()}')
print(f'GK train NaNs       : {gk_train_rolled[ROLLING_COLS_GK].isna().sum().sum()}')

Outfield train NaNs : 33588
GK train NaNs       : 2314


In [126]:
# Drop first appearance rows that have NaN from shift(1)
outfield_train_rolled = outfield_train_rolled.dropna(subset=ROLLING_COLS_OUTFIELD)
outfield_test_rolled  = outfield_test_rolled.dropna(subset=ROLLING_COLS_OUTFIELD)
gk_train_rolled       = gk_train_rolled.dropna(subset=ROLLING_COLS_GK)
gk_test_rolled        = gk_test_rolled.dropna(subset=ROLLING_COLS_GK)

print(f'Outfield train after dropping NaN rows : {outfield_train_rolled.shape[0]:,}')
print(f'Outfield test after dropping NaN rows  : {outfield_test_rolled.shape[0]:,}')
print(f'GK train after dropping NaN rows       : {gk_train_rolled.shape[0]:,}')
print(f'GK test after dropping NaN rows        : {gk_test_rolled.shape[0]:,}')

print(f'\nOutfield train NaNs remaining : {outfield_train_rolled[ROLLING_COLS_OUTFIELD].isna().sum().sum()}')
print(f'GK train NaNs remaining       : {gk_train_rolled[ROLLING_COLS_GK].isna().sum().sum()}')

Outfield train after dropping NaN rows : 21,847
Outfield test after dropping NaN rows  : 7,034
GK train after dropping NaN rows       : 2,193
GK test after dropping NaN rows        : 713

Outfield train NaNs remaining : 0
GK train NaNs remaining       : 0


In [127]:
sample = (
    outfield[['position_id', 'player_name']]
    .drop_duplicates(['position_id', 'player_name'])
    .groupby('position_id')['player_name']
    .apply(lambda x: list(x[:5]))
    .reset_index()
)
for _, row in sample.iterrows():
    print(f"{row['position_id']:>6} — {row['player_name']}")

sample = (
    gk[['position_id', 'player_name']]
    .drop_duplicates(['position_id', 'player_name'])
    .groupby('position_id')['player_name']
    .apply(lambda x: list(x[:5]))
    .reset_index()
)
for _, row in sample.iterrows():
    print(f"{row['position_id']:>6} — {row['player_name']}")

   0.0 — ['Adam Lallana', 'Junior Firpo', 'Thiago Silva', 'Mateo Kovacic', 'Diogo Jota']
  31.0 — ['Jacob Murphy', 'Javier Manquillo', 'Alex Iwobi', 'Connor Roberts', 'Tariq Lamptey']
  32.0 — ['Jurrien Timber', 'Michael Kayode', 'Matthew Lowton', 'Adam Webster', 'Reece James']
  33.0 — ['Jaydee Canvot', 'Taylor Harwood-Bellis', 'Matt Doherty', 'Emil Krafth', 'Joao Palhinha']
  34.0 — ['Kristoffer Vassbakk Ajer', 'Cristhian Mosquera', 'Shane Duffy', 'James Tarkowski', 'Trevoh Chalobah']
  35.0 — ['Maxence Lacroix', 'Nathan Wood', 'Emmanuel Agbadou', 'Federico Fernández', 'Cristian Romero']
  36.0 — ['Gabriel', 'Sepp van den Berg', 'Lewis Dunk', 'Ben Mee', 'Benoit Badiashile']
  37.0 — ['Chris Richards', 'Jack Stephens', 'Toti Gomes', 'Ciaran Clark', 'Radu Dragusin']
  38.0 — ['Rico Henry', 'Piero Hincapié', 'Pascal Gross', 'Charlie Taylor', 'Marc Cucurella']
  39.0 — ['Matt Ritchie', 'Vitalii Mykolenko', 'Charlie Taylor', 'Solly March', 'Rico Henry']
  51.0 — ['Aaron Wan-Bissaka', 'Kel

In [128]:
# ── Position group mapping ─────────────────────────────────────
POSITION_MAP = {
    # Defenders
    32.0: 'defender', 33.0: 'defender', 34.0: 'defender', 35.0: 'defender',
    36.0: 'defender', 37.0: 'defender', 38.0: 'defender', 39.0: 'defender',
    51.0: 'defender', 52.0: 'defender', 58.0: 'defender', 59.0: 'defender',
    62.0: 'defender', 68.0: 'defender', 71.0: 'defender', 72.0: 'defender',
    78.0: 'defender', 79.0: 'defender',
    # Midfielders
    31.0: 'midfielder', 63.0: 'midfielder', 64.0: 'midfielder', 65.0: 'midfielder',
    66.0: 'midfielder', 67.0: 'midfielder', 73.0: 'midfielder', 74.0: 'midfielder',
    75.0: 'midfielder', 76.0: 'midfielder', 77.0: 'midfielder', 82.0: 'midfielder',
    84.0: 'midfielder', 85.0: 'midfielder', 86.0: 'midfielder', 94.0: 'midfielder',
    95.0: 'midfielder',
    # Wingers
    83.0: 'winger', 87.0: 'winger', 88.0: 'winger', 92.0: 'winger',
    96.0: 'winger', 98.0: 'winger', 103.0: 'winger', 107.0: 'winger',
    # Forwards
    104.0: 'forward', 105.0: 'forward', 106.0: 'forward',
    114.0: 'forward', 115.0: 'forward', 116.0: 'forward',
    # Mixed / unclear
    0.0: 'midfielder',
}

# Add position_group column
outfield_train_rolled['position_group'] = outfield_train_rolled['position_id'].map(POSITION_MAP)
outfield_test_rolled['position_group']  = outfield_test_rolled['position_id'].map(POSITION_MAP)

# Check for any unmapped position IDs
unmapped_train = outfield_train_rolled[outfield_train_rolled['position_group'].isna()]['position_id'].unique()
unmapped_test  = outfield_test_rolled[outfield_test_rolled['position_group'].isna()]['position_id'].unique()
print(f'Unmapped position IDs in train : {unmapped_train}')
print(f'Unmapped position IDs in test  : {unmapped_test}')

print(f'\nPosition group distribution (train):')
print(outfield_train_rolled['position_group'].value_counts())

Unmapped position IDs in train : []
Unmapped position IDs in test  : []

Position group distribution (train):
position_group
defender      9548
midfielder    6764
winger        3110
forward       2425
Name: count, dtype: int64


In [129]:
gk_train_rolled['position_group'] = 'goalkeeper'
gk_test_rolled['position_group']  = 'goalkeeper'

print(f'GK train position_group : {gk_train_rolled["position_group"].unique()}')
print(f'GK test position_group  : {gk_test_rolled["position_group"].unique()}')
print(f'GK train rows           : {len(gk_train_rolled):,}')
print(f'GK test rows            : {len(gk_test_rolled):,}')

GK train position_group : ['goalkeeper']
GK test position_group  : ['goalkeeper']
GK train rows           : 2,193
GK test rows            : 713


In [130]:
from sklearn.preprocessing import RobustScaler
import joblib
from pathlib import Path

MODELS_DIR = Path('data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Columns to scale
SCALE_COLS_OUTFIELD = [c for c in outfield_train_rolled.columns
                       if c not in ID_COLS + ['rating_title', 'season', 'round', 
                                              'match_date', 'position_group']]

SCALE_COLS_GK = [c for c in gk_train_rolled.columns
                 if c not in ID_COLS + ['rating_title', 'season', 'round',
                                        'match_date', 'position_group']]

POSITION_GROUPS = ['defender', 'midfielder', 'winger', 'forward']

# ── Fit one scaler per position group on TRAIN only ───────────
scalers_outfield = {}

for group in POSITION_GROUPS:
    mask_train = outfield_train_rolled['position_group'] == group
    scaler     = RobustScaler()
    scaler.fit(outfield_train_rolled.loc[mask_train, SCALE_COLS_OUTFIELD])
    scalers_outfield[group] = scaler
    print(f'Fitted scaler for {group:<12} — {mask_train.sum():,} train rows')

# ── Fit GK scaler ─────────────────────────────────────────────
scaler_gk = RobustScaler()
scaler_gk.fit(gk_train_rolled[SCALE_COLS_GK])
print(f'Fitted scaler for goalkeeper  — {len(gk_train_rolled):,} train rows')

# ── Apply scalers to BOTH train and test ──────────────────────
for group in POSITION_GROUPS:
    mask_train = outfield_train_rolled['position_group'] == group
    mask_test  = outfield_test_rolled['position_group']  == group
    scaler     = scalers_outfield[group]

    outfield_train_rolled.loc[mask_train, SCALE_COLS_OUTFIELD] = scaler.transform(
        outfield_train_rolled.loc[mask_train, SCALE_COLS_OUTFIELD]
    )
    outfield_test_rolled.loc[mask_test, SCALE_COLS_OUTFIELD] = scaler.transform(
        outfield_test_rolled.loc[mask_test, SCALE_COLS_OUTFIELD]
    )

gk_train_rolled[SCALE_COLS_GK] = scaler_gk.transform(gk_train_rolled[SCALE_COLS_GK])
gk_test_rolled[SCALE_COLS_GK]  = scaler_gk.transform(gk_test_rolled[SCALE_COLS_GK])

# ── Save scalers ──────────────────────────────────────────────
joblib.dump(scalers_outfield, MODELS_DIR / 'scalers_outfield.pkl')
joblib.dump(scaler_gk,        MODELS_DIR / 'scaler_gk.pkl')
joblib.dump(POSITION_MAP, MODELS_DIR / 'position_map.pkl')

print(f'\nScalers saved to {MODELS_DIR.resolve()}')

Fitted scaler for defender     — 9,548 train rows
Fitted scaler for midfielder   — 6,764 train rows
Fitted scaler for winger       — 3,110 train rows
Fitted scaler for forward      — 2,425 train rows
Fitted scaler for goalkeeper  — 2,193 train rows

Scalers saved to C:\Users\natmaw\NatMaws Documents\Boston Stuff\EECE 5644 Machine Learning and Pattern Recognition\Final Project\GOALS-Game_Outcome_and_Analytics_Learning_System-\data\models


In [131]:
print('=== Scaled stats per position group ===\n')
for group in POSITION_GROUPS:
    mask = outfield_train_rolled['position_group'] == group
    stats = outfield_train_rolled.loc[mask, SCALE_COLS_OUTFIELD].agg(['min', 'median', 'std', 'max'])
    print(f'── {group.upper()} ──')
    print(f'  {"column":<30} {"min":>8} {"median":>8} {"std":>8} {"max":>8}')
    print(f'  {"-"*64}')
    for col in SCALE_COLS_OUTFIELD:
        print(f'  {col:<30} {stats.loc["min", col]:>8.3f} {stats.loc["median", col]:>8.3f} {stats.loc["std", col]:>8.3f} {stats.loc["max", col]:>8.3f}')
    print()

print('── GOALKEEPER ──')
stats_gk = gk_train_rolled[SCALE_COLS_GK].agg(['min', 'median', 'std', 'max'])
print(f'  {"column":<30} {"min":>8} {"median":>8} {"std":>8} {"max":>8}')
print(f'  {"-"*64}')
for col in SCALE_COLS_GK:
    print(f'  {col:<30} {stats_gk.loc["min", col]:>8.3f} {stats_gk.loc["median", col]:>8.3f} {stats_gk.loc["std", col]:>8.3f} {stats_gk.loc["max", col]:>8.3f}')

print(f'\nrating_title untouched — mean: {outfield_train_rolled["rating_title"].mean():.3f}')
print(f'\nAny NaNs in outfield train : {outfield_train_rolled[SCALE_COLS_OUTFIELD].isna().sum().sum()}')
print(f'Any NaNs in GK train       : {gk_train_rolled[SCALE_COLS_GK].isna().sum().sum()}')

=== Scaled stats per position group ===

── DEFENDER ──
  column                              min   median      std      max
  ----------------------------------------------------------------
  ShotsOffTarget                   -0.500    0.000    0.748   12.000
  ShotsOnTarget                     0.000    0.000    1.237   15.000
  accurate_passes                  -1.540    0.000    0.725    3.283
  assists                           0.000    0.000    0.126    2.000
  chances_created                  -0.667    0.000    0.986    9.333
  defensive_actions                -1.941    0.000    0.729    3.353
  expected_assists                 -0.529    0.000    0.991   15.941
  expected_goals                   -0.515    0.000    1.077   16.455
  goals                             0.000    0.000    0.108    1.200
  blocked_shots                    -1.000    0.000    1.332   14.000
  accurate_crosses                 -0.500    0.000    1.419   14.500
  dispossessed                     -0.333    0.00

In [132]:
GK_CLIP_COLS = [
    'aerial_win_rate',
    'ground_duel_win_rate',
    'goals_prevented',
    'save_rate',
]

for col in GK_CLIP_COLS:
    mean  = gk_train_rolled[col].mean()
    std   = gk_train_rolled[col].std()
    lower = mean - 5 * std
    upper = mean + 5 * std
    
    before_train = (gk_train_rolled[col] < lower).sum() + (gk_train_rolled[col] > upper).sum()
    before_test  = (gk_test_rolled[col] < lower).sum()  + (gk_test_rolled[col] > upper).sum()
    
    gk_train_rolled[col] = gk_train_rolled[col].clip(lower, upper)
    gk_test_rolled[col]  = gk_test_rolled[col].clip(lower, upper)
    
    print(f'{col}:')
    print(f'  Bounds         : [{lower:.3f}, {upper:.3f}]')
    print(f'  Rows clipped train : {before_train}')
    print(f'  Rows clipped test  : {before_test}')
    print(f'  New range train    : [{gk_train_rolled[col].min():.3f}, {gk_train_rolled[col].max():.3f}]')
    print()

aerial_win_rate:
  Bounds         : [-15.414, 14.402]
  Rows clipped train : 10
  Rows clipped test  : 2
  New range train    : [-15.414, 2.000]

ground_duel_win_rate:
  Bounds         : [-9.839, 9.861]
  Rows clipped train : 9
  Rows clipped test  : 3
  New range train    : [-9.839, 5.000]

goals_prevented:
  Bounds         : [-4.815, 4.895]
  Rows clipped train : 1
  Rows clipped test  : 3
  New range train    : [-4.815, 3.861]

save_rate:
  Bounds         : [-4.189, 4.106]
  Rows clipped train : 6
  Rows clipped test  : 3
  New range train    : [-4.189, 1.966]



In [133]:
# Export scaled train and test sets
import os

PROCESSED_DIR_OUTPUT = PROCESSED_DIR / 'datasets'
PROCESSED_DIR_OUTPUT.mkdir(parents=True, exist_ok=True)


outfield_train_rolled.to_parquet(PROCESSED_DIR_OUTPUT / 'outfield_train_scaled.parquet', index=False)
outfield_test_rolled.to_parquet(PROCESSED_DIR_OUTPUT  / 'outfield_test_scaled.parquet',  index=False)
gk_train_rolled.to_parquet(PROCESSED_DIR_OUTPUT       / 'gk_train_scaled.parquet',       index=False)
gk_test_rolled.to_parquet(PROCESSED_DIR_OUTPUT        / 'gk_test_scaled.parquet',        index=False)

# Confirm
print('=== Saved to data/processed/ ===')
print(f'  outfield_train_scaled.parquet — {len(outfield_train_rolled):,} rows x {outfield_train_rolled.shape[1]} cols')
print(f'  outfield_test_scaled.parquet  — {len(outfield_test_rolled):,} rows x {outfield_test_rolled.shape[1]} cols')
print(f'  gk_train_scaled.parquet       — {len(gk_train_rolled):,} rows x {gk_train_rolled.shape[1]} cols')
print(f'  gk_test_scaled.parquet        — {len(gk_test_rolled):,} rows x {gk_test_rolled.shape[1]} cols')

=== Saved to data/processed/ ===
  outfield_train_scaled.parquet — 21,847 rows x 57 cols
  outfield_test_scaled.parquet  — 7,034 rows x 57 cols
  gk_train_scaled.parquet       — 2,193 rows x 47 cols
  gk_test_scaled.parquet        — 713 rows x 47 cols
